# SafeSearch classifier tester
Try `safety.is_adult()` on your own text, a set of labeled cases, a stored Vespa document, or a live sample of FineWeb. Edit `safety.py` (term lists, `THRESHOLD`) and re-run the first cell to pick up changes.

In [ ]:
import importlib
import re
import pandas as pd
import safety

importlib.reload(safety)  # re-run this cell after editing safety.py
print("threshold:", safety.THRESHOLD)

## Explain a score
`explain()` shows which terms and URL hints add to the score.

In [ ]:
def explain(text, url=""):
    score = safety.adult_score(text, url)
    print(f"score = {score} / threshold {safety.THRESHOLD}  ->  adult = {safety.is_adult(text, url)}")
    return pd.DataFrame(safety.breakdown(text, url), columns=["term", "type", "count", "points"])

## 1. Try your own text

In [ ]:
text = """Paste the page text here"""
url = "https://example.com/page"  # optional, the URL adds to the score

explain(text, url)

## 2. Labeled test cases
Add your own rows. `expected` is what you want the filter to say.

In [ ]:
cases = [
    # (name, text, url, expected_adult)
    ("porn chatbot", "Get naughty with chat ai copilot! sexting ai porn companion ai sex chatbot ai erotic chat virtual sex partner ai porn free", "https://aimour.ai/chat-ai-copilot", True),
    ("porn keyword spam", "Teen Lesbian - Best New Porn, Free Porn, Porn Videos, Porn Movies, Best Porn, Sex Movies Download, Porn Videos Download, Porn, Free Mobile Porn. Hi, my name is Emily and I love sports.", "https://www.teen-lesbian.com/EmelyHoward/", True),
    ("porn site by url", "Watch free videos", "https://realincestporn.pro/en/video/1", True),
    ("fetish stuffing", "foot fetish chat " + "fetish " * 12, "https://example.com/chat", True),
    ("JAV video site", "Tawaman's wifey gives great deepthroat jobs with squirting orgasm. This JAV Gonzo video: she likes using dildos. gets two cocks in her throat in a super-fucking-hot three female fuck festival in this JAV porn! Very first time threesome. the best creampies in all of Asia.", "https://perrihdvideo.pro/vidse/x/", True),
    ("profanity only", "Fuck this, fucking traffic again. Fucked up day, what the fuck. Fuck it, I quit.", "https://example.com/rant", False),
    ("medical anatomy", "The vagina and penis are parts of the reproductive system. Vagina health and penis health are covered in this textbook chapter.", "https://example.com/anatomy", False),
    ("platform rules", "Medium rules: no sexual content, no nude images, report harassment.", "https://policy.medium.com/medium-rules", False),
    ("news: harassment", "Former assistant sues for sexual harassment, court filing says", "https://example.com/news", False),
    ("sex education", "Sex education: sex, contraception and sexual health for teens", "https://example.com/health", False),
    ("Ford Escort", "The Ford Escort is a small car", "https://example.com/ford-escort", False),
    ("harmless words", "Data analysis of the cockpit and Scunthorpe", "https://example.com/a", False),
    ("colour named nude", "Tinted lip balm in shade nude. Nude pink, nude beige, nude tones, nude finish.", "https://example.com/lip-balm", False),
    ("xxx placeholder", "Dial +xxx xxx xxx xxx for the country code of the USA", "https://example.com/country-code", False),
    ("Pussy Riot news", "Pussy Riot takes Phoenix by storm; the Pussy Riot collective said the Pussy Riot tour sold out; Pussy Riot fans queued.", "https://example.com/pussy-riot", False),
]

df = pd.DataFrame(
    [(n, safety.adult_score(t, u), safety.is_adult(t, u), exp) for n, t, u, exp in cases],
    columns=["case", "score", "is_adult", "expected"],
)
df["ok"] = df["is_adult"] == df["expected"]
print(f"{df['ok'].sum()}/{len(df)} correct")
df

## 3. Check a stored Vespa document
Looks up documents whose url contains a fragment and compares the stored `adult` flag with what `is_adult()` says now. If they differ, run `python update_adult.py`.

In [ ]:
from config import connect_vespa

vespa_application = connect_vespa()  # local/cloud from VESPA_MODE in .env
fragment = "teen-lesbian"

r = vespa_application.query(yql=f"select * from sources * where url contains '{fragment}' limit 5")
pd.DataFrame(
    [
        {
            "url": h["fields"]["url"][:70],
            "stored_adult": h["fields"].get("adult"),
            "score_now": safety.adult_score(h["fields"]["text"], h["fields"]["url"]),
            "is_adult_now": safety.is_adult(h["fields"]["text"], h["fields"]["url"]),
        }
        for h in r.hits
    ]
)

## 4. Live sample of FineWeb
Streams a few thousand documents, shows how many get flagged, which ones, and near misses just under the threshold. Good for tuning.

In [ ]:
import os
from config import load_env
from datasets import load_dataset

load_env()
N = 2000  # documents to sample

stream = load_dataset("HuggingFaceFW/fineweb", "CC-MAIN-2025-08", split="train", streaming=True)
rows = [
    {"score": safety.adult_score(d["text"], d["url"]), "url": d["url"], "tokens": d["token_count"]}
    for d in stream.take(N)
]
sample = pd.DataFrame(rows)
flagged = sample[sample.score >= safety.THRESHOLD].sort_values("score", ascending=False)
near = sample[(sample.score >= safety.THRESHOLD // 2) & (sample.score < safety.THRESHOLD)].sort_values("score", ascending=False)
print(f"{len(flagged)}/{len(sample)} flagged ({100 * len(flagged) / len(sample):.2f}%)")
flagged

In [ ]:
# Near misses: just under the threshold, check for pages that should have been flagged
near.head(20)